# Tutorial #1 - MLOps Workflow

This notebook demonstrates the manual DVC + Hydra + MLflow integration pattern:

1. Load the Jena Climate dataset (already tracked by DVC)
2. Read the DVC content hash from the `.dvc` pointer file
3. Load experiment configuration via Hydra and compute its hash
4. Log both hashes to MLflow for strict data and configuration lineage
5. Train a simple baseline model and log metrics

All steps use standard Python code and CLI tools - no platform-specific features.

In [1]:
import os
import keras
from zipfile import ZipFile
import tqdm as notebook_tqdm
import pandas as pd
import numpy as np
import yaml
import hashlib

from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf

import mlflow
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

I0000 00:00:1773759300.523575    2346 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773759300.552620    2346 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1773759301.300230    2346 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


/app/data/environments/python/3.12/venv_jena_weather/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Data Ingestion

The Jena Climate dataset is hosted in the TensorFlow/Keras dataset repository. The ingestion script downloads the compressed file, extracts it, and places the CSV in the `data/` folder. Once ingested, we track it with DVC to enable reproducible versioning.

In [2]:
data_dir = os.path.join(os.getcwd(), "data")
csv_path = os.path.join(data_dir, "jena_climate_2009_2016.csv")

if not os.path.exists(csv_path):
    uri = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip"
    zip_path = keras.utils.get_file(origin=uri, fname="jena_climate_2009_2016.csv.zip")
    ZipFile(zip_path).extractall(data_dir)
    print(f"Downloaded and extracted to {csv_path}")
else:
    print(f"Dataset already exists at {csv_path}")

print(f"File size: {os.path.getsize(csv_path) / 1024 / 1024:.1f} MB")

Dataset already exists at /app/mounts/jena_weather/data/jena_climate_2009_2016.csv
File size: 41.2 MB


### DVC Tracking

After ingestion, we track the dataset with DVC. This creates a `.dvc` pointer file containing the MD5 hash and adds the CSV to `.gitignore` (since DVC manages it via the remote storage, not Git).

In [3]:
!dvc add data/jena_climate_2009_2016.csv
!git add data/jena_climate_2009_2016.csv.dvc data/.gitignore
!dvc push

⠋ Checking graph
Adding...                                                                       
!
                                                                                
!
  0% Checking cache in '/app/mounts/jena_weather/.dvc/cache/files/md5'| |0/? [00
                                                                                
!
  0%|          |Checking out /app/mounts/jena_weather/0/1 [00:00<?,    ?files/s]


100% Adding...|████████████████████████████████████████|1/1 [00:00, 27.11file/s]

To track the changes with git, run:

	git add data/jena_climate_2009_2016.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true


Pushing
!
  0% Checking cache in 'noted-dvc/files/md5'|        |0/? [00:00<?,    ?files/s]


100% Querying cache in 'noted-dvc/files/md5'|███|1/1 [00:00<00:00,  4.92files/s]
Pushing
Everything is up to date.


## 2. Dataset Loading

The Jena Climate dataset contains 420,551 observations across 15 columns (14 meteorological variables plus a timestamp), recorded every 10 minutes from January 2009 to December 2016.

In [4]:
df = pd.read_csv(csv_path)

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Date range: {df['Date Time'].iloc[0]} to {df['Date Time'].iloc[-1]}")
df.head()

Shape: (420551, 15)
Columns: ['Date Time', 'p (mbar)', 'T (degC)', 'Tpot (K)', 'Tdew (degC)', 'rh (%)', 'VPmax (mbar)', 'VPact (mbar)', 'VPdef (mbar)', 'sh (g/kg)', 'H2OC (mmol/mol)', 'rho (g/m**3)', 'wv (m/s)', 'max. wv (m/s)', 'wd (deg)']
Date range: 01.01.2009 00:10:00 to 01.01.2017 00:00:00


,Date Time,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),rho (g/m**3),wv (m/s),max. wv (m/s),wd (deg)
0,01.01.2009 00:10:00,996.52,-8.02,265.40,-8.90,93.3,3.33,3.11,0.22,1.94,3.12,1307.75,1.03,1.75,152.3
1,01.01.2009 00:20:00,996.57,-8.41,265.01,-9.28,93.4,3.23,3.02,0.21,1.89,3.03,1309.80,0.72,1.50,136.1
2,01.01.2009 00:30:00,996.53,-8.51,264.91,-9.31,93.9,3.21,3.01,0.20,1.88,3.02,1310.24,0.19,0.63,171.6
3,01.01.2009 00:40:00,996.51,-8.31,265.12,-9.07,94.2,3.26,3.07,0.19,1.92,3.08,1309.19,0.34,0.50,198.0
4,01.01.2009 00:50:00,996.51,-8.27,265.15,-9.04,94.1,3.27,3.08,0.19,1.92,3.09,1309.00,0.32,0.63,214.3


## 3. DVC Data Lineage

The `.dvc` pointer file contains the MD5 hash that uniquely identifies this exact version of the data. We read this hash and will log it into MLflow to establish strict data lineage - every experiment run is linked to the precise dataset version used.

In [5]:
dvc_file = csv_path + ".dvc"

with open(dvc_file) as f:
    dvc_meta = yaml.safe_load(f)

dvc_hash = dvc_meta["outs"][0]["md5"]
dvc_size = dvc_meta["outs"][0]["size"]

print(f"DVC MD5 hash: {dvc_hash}")
print(f"File size:    {dvc_size:,} bytes ({dvc_size / 1024 / 1024:.1f} MB)")

DVC MD5 hash: 959915f05bfafef18e471a97ae679535
File size:    43,164,220 bytes (41.2 MB)


## 4. Preprocessing

Following the project guidelines, we select 6 input features (avoiding near-deterministic relationships with the target), resample to hourly frequency, and split temporally.

In [6]:
# Parse dates and set as index
df["Date Time"] = pd.to_datetime(df["Date Time"], format="%d.%m.%Y %H:%M:%S")
df.set_index("Date Time", inplace=True)

# Replace erroneous wind speed values
df[["wv (m/s)", "max. wv (m/s)"]] = df[["wv (m/s)", "max. wv (m/s)"]].replace(-9999.0, np.nan)

# Remove duplicates
n_before = len(df)
df = df[~df.index.duplicated(keep="first")]
print(f"Removed {n_before - len(df)} duplicate rows")

# Resample to hourly (mean aggregation)
df = df.resample("1h").mean().dropna()
print(f"Hourly shape: {df.shape}")

# Select features
feature_cols = ["T (degC)", "p (mbar)", "rh (%)", "wv (m/s)", "max. wv (m/s)", "wd (deg)"]
target_col = "T (degC)"
df_model = df[feature_cols].copy()

# Temporal split (70% train, 15% val, 15% test)
n = len(df_model)
train_end = int(n * 0.7)
val_end = int(n * 0.85)

df_train = df_model.iloc[:train_end]
df_val = df_model.iloc[train_end:val_end]
df_test = df_model.iloc[val_end:]

print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")

Removed 327 duplicate rows
Hourly shape: (70038, 14)
Train: 49026, Val: 10506, Test: 10506


## 5. Hydra Configuration

We use Hydra to manage experiment configuration centrally. The `config/config.yaml` file defines all parameters (data paths, features, model type, split ratios) in one place. We load the resolved config, compute its hash, and will log it to MLflow alongside the DVC hash - establishing both data and configuration lineage.

In [7]:
config_dir = os.path.join(os.getcwd(), "config")
with initialize_config_dir(config_dir=config_dir, version_base=None):
    cfg = compose(config_name="config")

config_yaml = OmegaConf.to_yaml(cfg)
config_hash = hashlib.md5(config_yaml.encode()).hexdigest()

print("Resolved Hydra config:")
print(config_yaml)
print(f"Config MD5 hash: {config_hash}")

Resolved Hydra config:
data:
  file: data/jena_climate_2009_2016.csv
  resample_freq: 1h
  features:
  - T (degC)
  - p (mbar)
  - rh (%)
  - wv (m/s)
  - max. wv (m/s)
  - wd (deg)
  target: T (degC)
  split:
    train: 0.7
    val: 0.15
    test: 0.15
model:
  type: LinearRegression
forecast:
  strategy: 1-step ahead

Config MD5 hash: 5644215816830469172a413ae7b53c07


## 6. Baseline Model with MLflow Tracking

We train a simple linear regression as a baseline and log everything to MLflow: the DVC data hash (as both parameter and tag), model parameters, and evaluation metrics. This establishes the data lineage pattern that will be used with more complex models (GRU, PatchTST).

In [8]:
# Prepare features and target (1-step ahead: predict next hour's temperature)
X_train = df_train[feature_cols].iloc[:-1].values
y_train = df_train[target_col].iloc[1:].values

X_test = df_test[feature_cols].iloc[:-1].values
y_test = df_test[target_col].iloc[1:].values

with mlflow.start_run(run_name="baseline_linear_regression"):
    # Log DVC data hash - strict data lineage
    mlflow.log_param("dvc_data_hash", dvc_hash)
    mlflow.set_tag("dvc.data_hash", dvc_hash)
    mlflow.set_tag("dvc.data_file", "data/jena_climate_2009_2016.csv")

    # Log Hydra config hash - configuration lineage
    mlflow.log_param("hydra_config_hash", config_hash)
    mlflow.set_tag("hydra.config_hash", config_hash)
    mlflow.log_text(config_yaml, "hydra_config.yaml")

    # Log experiment parameters
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("features", feature_cols)
    mlflow.log_param("target", target_col)
    mlflow.log_param("train_size", len(X_train))
    mlflow.log_param("test_size", len(X_test))
    mlflow.log_param("forecast_strategy", "1-step ahead")

    # Train
    model = LinearRegression()
    model.fit(X_train, y_train)

    # Evaluate
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)

    print(f"MAE:  {mae:.4f} C")
    print(f"RMSE: {rmse:.4f} C")
    print(f"DVC hash logged: {dvc_hash}")

MAE:  0.6410 C
RMSE: 0.8842 C
DVC hash logged: 959915f05bfafef18e471a97ae679535
🏃 View run baseline_linear_regression at: http://mlflow:5000/#/experiments/3/runs/d8ebf45b71ab4ecc9637776d483bfe2d
🧪 View experiment at: http://mlflow:5000/#/experiments/3


## 7. Verify Data Lineage in MLflow

We confirm the DVC hash was correctly logged by querying the last run.

In [9]:
runs = mlflow.search_runs(order_by=["start_time DESC"], max_results=1)
run = runs.iloc[0]

print(f"Run:           {run['tags.mlflow.runName']}")
print(f"DVC hash (param): {run['params.dvc_data_hash']}")
print(f"DVC hash (tag):   {run['tags.dvc.data_hash']}")
print(f"Data file:        {run['tags.dvc.data_file']}")
print(f"Config hash:      {run['tags.hydra.config_hash']}")
print(f"MAE:              {run['metrics.mae']:.4f}")
print(f"RMSE:             {run['metrics.rmse']:.4f}")

Run:           baseline_linear_regression
DVC hash (param): 959915f05bfafef18e471a97ae679535
DVC hash (tag):   959915f05bfafef18e471a97ae679535
Data file:        data/jena_climate_2009_2016.csv
Config hash:      5644215816830469172a413ae7b53c07
MAE:              0.6410
RMSE:             0.8842
